# Sprint 3 · Promote and recover the selected model

Linden Facilities needs 24 hourly predictions as one file within 60 seconds. Continue in the same repository and `work/`; the saved experiment plan and selected run carry into later sprints. This is an optional exploration notebook. [Campus lessons](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/README.md) also provide terminal/API routes. Keep one set of work. There is nothing to submit.

Follow [local setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md), select the project's Python 3.12 kernel, and run setup. Restarting a kernel loses variables, not saved run IDs. Do not reset the project between sections. Independent cells are for your attempt; guided outputs alone do not answer them.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Use Python 3.12. Locally run uv sync --locked --python 3.12 and select .venv/bin/python.')
# Hosted exploration is optional. Local setup uses the already installed editable project.
try:
    import linden_mlops
except ImportError:
    if 'google.colab' not in sys.modules:
        raise RuntimeError('Follow docs/setup.md from the repository root, then select its .venv kernel.')
    hosted_root = Path('/content/ml-llm-ops')
    if not hosted_root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/richhiey/ml-llm-ops.git', str(hosted_root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(hosted_root), 'pytest==8.4.2'], check=True)
    sys.path.insert(0, str(hosted_root / 'src'))
    import linden_mlops
from linden_mlops import (Config, Workspace, ingest, preprocess, train, evaluate, run_pipeline,
                          export_runs, verify_summary, register, approve, activate, batch_predict, reproduce)
from linden_mlops.lifecycle import ROOT, MODEL_NAME, code_revision
from linden_mlops.project import Project
from linden_mlops.dev import course_cli, project_tests, project_python, prepare_case
import pandas as pd
import numpy as np
progress = Project()
project = PACKAGE = ROOT
state = progress.work
ws = Workspace(progress.tracking)
print('Working source:', ROOT)
print('Retained work:', state)
print('Code:', code_revision())
print('Saved decisions:', sorted(progress.data))

## 1 · Separate artifact identity from active selection

A registered model groups versions. Each version points to one source run and artifact; tags hold validation evidence; the mutable `active` alias chooses what the next job resolves. Registering alone does not approve or deploy a model. Old staging/production/archived stage labels are superseded vocabulary: reason with aliases, tags and environment-specific registered models.

| Evidence | Question |
|---|---|
| Run -> model artifact | What execution produced it? |
| Version + validation tags | Which artifact was checked? |
| Active alias | What will a new job resolve? |
| Retained alias event | What changed, when and why? |

Continue with the exact saved validation choice, not a fresh alpha `1` baseline.

In [ ]:
selection = progress.selected()
print(selection)
selected_run = ws.client.get_run(selection['run_id'])
assert selected_run.data.params['evaluation_partition'] == 'validation'

## 2 · Register, validate and select that artifact

Run the first cell to register the selected artifact. The saved registry identity is reused on reopening. On its first execution, activation while pending must fail. This boundary is our local application's policy, not MLflow authorization. Approval does not itself execute tests.

In [ ]:
if not progress.data.get('registry'):
    version = register(ws, selection['run_id'])
    progress.save(registry={'version': version, 'run_id': selection['run_id']})
version = progress.require('registry')['version']
mv = ws.client.get_model_version(MODEL_NAME, version)
if mv.tags.get('validation_status') == 'pending':
    try:
        activate(ws, version, 'Deliberate pending probe')
    except ValueError as error:
        print('Expected rejection:', error)
    else:
        raise AssertionError('Pending version unexpectedly activated')
else:
    print('Reusing an already checked version:', version, mv.tags)
assert mv.run_id == selection['run_id']

Run the contract test against saved source, inspect the chosen run's finite validation RMSE (at most `5` kWh), then record an honest reason and activate. The batch resolves that version once. Compare its source run and version with registration; current aliases cannot prove what an earlier job loaded.

In [ ]:
if not progress.data.get('promotion'):
    project_tests(ROOT, 'tests/test_spine.py::test_contract_extension_and_unchanged_pipeline', '-q')
    approve(ws, version, 'Observed passing input contract; guarded finite validation RMSE <= 5 kWh')
    event = activate(ws, version, 'Selected Sprint 2 configuration after validation checks')
    output, inference = batch_predict(ws)
    progress.save(promotion={'event': event, 'inference': inference})
promotion = progress.require('promotion')
assert promotion['inference']['run_id'] == selection['run_id']
assert promotion['inference']['resolved_version'] == version
print(promotion)
print(ws.client.get_model_version(MODEL_NAME, version).tags)

## 3 · Identify the recoverable recipe

A metric match alone is insufficient. The manifest ties code content, data bytes, config, evaluation partition, environment and fitted model to one run. The lock defines dependency resolution; the recorded environment also identifies interpreter, installed packages and platform. Retaining only a pickle cannot recover the training recipe.

Inspect the manifest and source archive in MLflow. Which field would reveal a changed split or a different code edit? Which identifier is expected to change when you rerun?

In [ ]:
recipe_path = Path(ws.client.download_artifacts(selection['run_id'], 'reproduction/manifest.json'))
recipe = json.loads(recipe_path.read_text())
print({key: recipe[key] for key in ['run_id', 'config', 'code_revision', 'data_sha256', 'evaluation_partition', 'model_uri']})
print('Environment:', recipe['environment'])

## 4 · Reproduce the past result

Use the recorded selected run and the `reproduce` API/CLI to rerun into a new store, then inspect `reproduction-check.json`. Check source, data, config, partition and environment before interpreting the absolute RMSE delta. The declared tolerance is `1e-6` kWh. Explain changed IDs versus invariant recipe fields.

Terminal route: `uv run linden-mlops reproduce --destination work/past-replay`. This is a same-environment replay, not yet the new-source/new-environment proof. Keep one replay and reuse it.

In [ ]:
# Your attempt: use reproduce(ws, selection['run_id'], state / 'past-replay')
# The destination must not exist. Inspect and explain the returned comparison.

## 5 · Investigate a rollback incident

The main cumulative project's selection stays intact. Create an isolated incident below once. The client now requires validation RMSE at most `1.6` kWh; a trial is active. Read actual version tags and raw source runs, identify the prior approved version that qualifies, and reassign its alias with a reason. Preparation does not perform a rollback.

Use the API lookup or `activate --state work/rollback --version ... --reason ...`, then exercise a new batch. Tag the trial incident and retain both artifacts. Explain the old/new versions, source run, timestamp and reason in `audit.jsonl`; this mutable helper-mediated log is not a tamper-proof transaction.

In [ ]:
incident_state = state / 'rollback'
if not incident_state.exists():
    prepare_case('rollback', incident_state)
incident = Workspace(incident_state)
print((incident_state / 'case.json').read_text())
display(export_runs(incident))
for item in incident.client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(item.version, item.run_id, item.tags, item.aliases)

In [ ]:
# Inspect before/after selection; choose the qualifying prior approved version.
# Activate it with an evidence-based reason, tag the trial incident, then batch_predict(incident).
# Compare the new job's run/version and retained audit event.

## 6 · Harbor brings a different incident

Prepare Harbor's intake once. Its revised limit is `1.45` kWh, and the newest artifact is pending. Determine the qualifying prior approved version using its actual evidence, restore it, tag the trial incident and exercise the next batch. Recover that version's run into a new store and check lineage plus tolerance. Reuse the main project's registration/promotion as your connected lifecycle example.

Keep all three artifacts. Explain why the highest version number is not the answer by itself and what your audit can/cannot establish. No new pipeline or written submission is needed.

In [ ]:
harbor_state = state / 'harbor-registry'
if not harbor_state.exists():
    prepare_case('registry-assessment', harbor_state)
harbor = Workspace(harbor_state)
print((harbor_state / 'case.json').read_text())
display(export_runs(harbor))
for item in harbor.client.search_model_versions(f"name='{MODEL_NAME}'"):
    print(item.version, item.run_id, item.tags, item.aliases)

In [ ]:
# Resolve Harbor's incident using taught operations and explicit IDs.
# Check the restored job and reproduce its source run into a new state.
# Preserve the main cumulative project's selection.